In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import torch

from GraphST import GraphST
from GraphST.preprocess import filter_with_overlap_gene
from GraphST.utils import project_cell_to_spot

## scRNA data
Download the supplementary table of [this paper](https://www.nature.com/articles/s41588-021-00911-1#Sec39).

In [ ]:
adata_sc = sc.read_h5ad("results_breast/scRNA.h5ad")
genes = pd.read_excel("41467_2021_26271_MOESM16_ESM.xlsx", header=None)[0].to_list()
genes_for_stsc = np.intersect1d(adata_sc.var_names, genes)

In [7]:
adata_sc_list = dict()
adata_sc_list['CID4290'] = adata_sc[adata_sc.obs["orig.ident"] == 'CID4290A', genes_for_stsc]
adata_sc_list['CID4535'] = adata_sc[adata_sc.obs["orig.ident"] == 'CID4535', genes_for_stsc]
adata_sc_list['CID4465'] = adata_sc[adata_sc.obs["orig.ident"] == 'CID4465', genes_for_stsc]
adata_sc_list['CID44971'] = adata_sc[adata_sc.obs["orig.ident"] == 'CID44971', genes_for_stsc]

In [8]:
sample_list = adata_sc_list.keys()
for sample in sample_list:
    adata_sc_list[sample].var_names_make_unique()
    GraphST.preprocess(adata_sc_list[sample])

In [9]:
adata_sc_list

{'CID4290': AnnData object with n_obs × n_vars = 5789 × 5504
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major'
     var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
     uns: 'hvg', 'log1p',
 'CID4535': AnnData object with n_obs × n_vars = 3961 × 5504
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major'
     var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
     uns: 'hvg', 'log1p',
 'CID4465': AnnData object with n_obs × n_vars = 1564 × 5504
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major'
     var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
     uns: 'hvg', 'log1p',
 'CID44971': 

## ST data

In [10]:
adata_list = dict()
sample_list = ['CID4535', 'CID4290', 'CID4465', 'CID44971']
for sample in sample_list:
    print(sample)
    adata_list[sample] = sc.read_h5ad("results_breast/breast_st_raw/{}.h5ad".format(sample))
adata_list

CID4535
CID4290
CID4465
CID44971


{'CID4535': AnnData object with n_obs × n_vars = 1016 × 16765
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR', 'n_counts'
     var: 'n_cells'
     uns: 'spatial'
     obsm: 'spatial'
     layers: 'raw_counts',
 'CID4290': AnnData object with n_obs × n_vars = 2398 × 17506
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR', 'n_counts'
     var: 'n_cells'
     uns: 'spatial'
     obsm: 'spatial'
     layers: 'raw_counts',
 'CID4465': AnnData object with n_obs × n_vars = 1207 × 17362
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR', 'n_counts'
     var: 'n_cells'
     uns: 'spatial'
     obsm: 'spatial'
     layers: 'raw_counts',
 'CID44971': AnnData object with n_obs × n_vars = 1125 × 17479
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER

## Pre-processing for ST data

In [11]:
for sample in sample_list:
    GraphST.preprocess(adata_list[sample])
    GraphST.construct_interaction(adata_list[sample])
    GraphST.add_contrastive_label(adata_list[sample])

## Finding overlap genes between ST and reference data

In [12]:
adata_sc_prep = dict()
adata_prep = dict()
for sample in sample_list:
    adata_sc = adata_sc_list[sample]
    adata = adata_list[sample]
    adata, adata_sc = filter_with_overlap_gene(adata, adata_sc)
    adata_sc_prep[sample] = adata_sc
    adata_prep[sample] = adata

Number of overlap genes: 923
Number of overlap genes: 808


C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:34: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["overlap_genes"] = genes
C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:35: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_sc.uns["overlap_genes"] = genes
C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:34: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["overlap_genes"] = genes
C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:35: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_sc.uns["overlap_genes"] = genes


Number of overlap genes: 897
Number of overlap genes: 887


C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:34: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["overlap_genes"] = genes
C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:35: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_sc.uns["overlap_genes"] = genes
C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:34: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["overlap_genes"] = genes
C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:35: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_sc.uns["overlap_genes"] = genes


## Extracting features for ST data

In [13]:
for sample in sample_list:
    GraphST.get_feature(adata_prep[sample])

adata_prep

C:\Users\linan\anaconda3\envs\graphst\Lib\site-packages\GraphST\preprocess.py:118: ImplicitModificationWarning: Setting element `.obsm['feat']` of view, initializing view as actual.
  adata.obsm['feat'] = feat


{'CID4535': AnnData object with n_obs × n_vars = 1016 × 923
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR', 'n_counts'
     var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
     uns: 'spatial', 'hvg', 'log1p', 'overlap_genes'
     obsm: 'spatial', 'distance_matrix', 'graph_neigh', 'adj', 'label_CSL', 'feat', 'feat_a'
     layers: 'raw_counts',
 'CID4290': AnnData object with n_obs × n_vars = 2398 × 808
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR', 'n_counts'
     var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
     uns: 'spatial', 'hvg', 'log1p', 'overlap_genes'
     obsm: 'spatial', 'distance_matrix', 'graph_neigh', 'adj', 'label_CSL', 'feat', 'feat_a'
     layers: 'raw_counts',
 'CID4465': AnnData object with n_obs × n_va

## Implementing GraphST for cell type deconvolution

In [14]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

adata_sc_trained = dict()
adata_trained = dict()
model_trained = dict()

for sample in sample_list:
    adata_sc = adata_sc_prep[sample]
    adata = adata_prep[sample]
    
    # Train model
    model = GraphST.GraphST(adata, adata_sc, epochs=1200, random_seed=50, device=device, deconvolution=True)
    adata, adata_sc = model.train_map()
    
    adata_sc_trained[sample] = adata_sc
    adata_trained[sample] = adata
    model_trained[sample] = model

Begin to train ST data...


100%|█████████████████████████████████████████████████████████████████████████████| 1200/1200 [00:09<00:00, 132.14it/s]


Optimization finished for ST data!
Begin to train scRNA data...


100%|█████████████████████████████████████████████████████████████████████████████| 1200/1200 [00:02<00:00, 553.84it/s]


Optimization finished for cell representation learning!
Begin to learn mapping matrix...


...

Optimization finished for ST data!
Begin to train scRNA data...


100%|█████████████████████████████████████████████████████████████████████████████| 1200/1200 [00:03<00:00, 316.21it/s]


Optimization finished for cell representation learning!
Begin to learn mapping matrix...


100%|█████████████████████████████████████████████████████████████████████████████| 1200/1200 [00:09<00:00, 124.54it/s]


Mapping matrix learning finished!


## Visualization of single cell data distribution in ST tissue

In [27]:
!mkdir results_breast/celltype_major
!mkdir results_breast/celltype_minor
!mkdir results_breast/celltype_subset

In [28]:
for celltype_name in ['celltype_major', 'celltype_minor', 'celltype_subset']:
    for sample in sample_list:
        adata_sc_trained[sample].obs['cell_type'] = adata_sc_trained[sample].obs[celltype_name]
        project_cell_to_spot(adata_trained[sample], adata_sc_trained[sample], retain_percent=0.15)
        celltypes = adata_sc_trained[sample].obs[celltype_name].unique()
        df = pd.DataFrame(index=adata_trained[sample].obs.index,
                         columns = celltypes)
        for celltype in celltypes:
            df[celltype] = adata_trained[sample].obs[celltype]
        df.to_csv('results_breast/{}/{}.csv'.format(celltype_name, sample))